# タンパク質言語モデルによる機械学習入門
## ProtBERT 埋め込み + シンプル分類器で膜貫通型タンパク質を予測

## 実習概要

本ノートブックでは、以下を学習します:

1. **タンパク質言語モデル(ProtBERT)**の仕組み
   - アミノ酸配列を数値ベクトル(埋め込み)に変換
   - 言語モデルが「タンパク質の言語」を学習していることを理解

2. **埋め込みベクトルの性質**
   - 次元削減と可視化(t-SNE)
   - 類似したタンパク質は埋め込み空間で近くなる

3. **機械学習による分類**
   - ロジスティック回帰
   - ランダムフォレスト
   - 両者の精度比較

## 分類タスク

**課題**: アミノ酸配列だけから、そのタンパク質が「膜貫通型(膜受容体など)」か「可溶性(血液中や細胞質に存在)」かを予測する

- **膜貫通型の例**: GPCR, チャネル蛋白, トランスポーター
- **可溶性の例**: 酵素, 抗体, ホルモン

膜貫通型タンパク質には「疎水性のアミノ酸が集中した領域」という特徴があり、ProtBERT はこれを埋め込みに自動的に符号化しています。

## 0. 必要ライブラリのインストール(初回のみ)

In [ ]:
# 初回のみ実行してください
# !pip install transformers torch scikit-learn pandas numpy matplotlib seaborn scipy requests

## 1. ライブラリのインポート

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, roc_curve
)
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import torch
from transformers import AutoTokenizer, AutoModel
import warnings

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

print("✓ ライブラリのインポート完了")

## 2. タンパク質言語モデルの準備

Hugging Face から事前学習済みの ProtBERT モデルをダウンロードします。初回は数秒かかります。

In [ ]:
print("ProtBERT をロード中 (初回は1分程度かかります)...")

model_name = "Rostlab/prot_bert"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

print(f"✓ ProtBERT ロード完了")
print(f"  デバイス: {device}")
print(f"  パラメータ数: {sum(p.numel() for p in model.parameters()):,}")

## 3. タンパク質配列 → 埋め込みベクトルへの変換

In [ ]:
def get_protein_embedding(sequence, tokenizer, model, device):
    """
    アミノ酸配列をProtBERT埋め込みに変換する関数
    """
    sequence_spaced = " ".join(sequence)
    inputs = tokenizer(sequence_spaced, return_tensors="pt", max_length=4096,
                      truncation=True).to(device)
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    embedding = outputs.last_hidden_state.mean(dim=1).squeeze().cpu().numpy()
    
    return embedding

# テスト実行
test_sequence = "MKTAYIAKQRQISFVKSHFSRQLEERLGLIEVQAPILSRVGDGTQDNLSGAEKAVQVK"
test_embedding = get_protein_embedding(test_sequence, tokenizer, model, device)

print(f"✓ 埋め込み変換完了")
print(f"  入力シーケンス長: {len(test_sequence)} アミノ酸")
print(f"  出力ベクトル次元: {len(test_embedding)}")
print(f"  埋め込みベクトル(先頭10次元): {test_embedding[:10]}")

## 4. 訓練データの準備

In [ ]:
transmembrane_proteins = {
    "beta2-adrenergic receptor": "MGQPGNGSAFLLAPNGSGLQWPHIIGQQPNGLSIGFQNGTSDVTAAYVQQVHNGVRWLF",
    "Mu opioid receptor": "MPPGNATAAGGPCPGLVLGVWVCSCVSIGVGYGQGHRSYSFSATRWQSAGVRSRCCCLA",
    "Voltage-gated potassium channel": "MSSGVAAGAGAGACGCMSCCCGALCCCCCATAGAGAGGPLSPTPACTDAPVPPPPTP",
    "Transferrin receptor": "MKSAYIAKQRQISFVKSHFSRQLEERLGLIEVQAPILSRVGDGTQDNLSGAEKAVQVK",
    "Glucose transporter 1": "MPPGPLPTHSQAQQGGQPQRARPPRPASAQCRAPPAAALPGPGHGSASDLPPPGAAP",
    "Rhodopsin": "MNGTEGPNFYVPFSNATGVVRSPFEYPQYYLAEPWQFSMLAAYMFLLIVLGFPINFLTLYVTVQHKKLR",
}

soluble_proteins = {
    "Lysozyme": "MKALAVALATVFADYKDDDKGDPVQWWVNGKFLNGPYAKFDDNKRQGKYGFNDNEKR",
    "Hemoglobin (alpha)": "MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSHGSAQVKGH",
    "Immunoglobulin G (Fab)": "DIQMTQSPSSLSASVGDRVTITCRASQSISSYLMHWVKQRPGKGLEWIGEIDPD",
    "Insulin": "GIVEQCCTSICSLYQLENYCN",
    "TNF-alpha": "MNLGHQSLGTLMALGLTALACGQDVKGWSVPAVPVAQSMYIFPAFSVDTCLVNVPVQL",
    "Carboxypeptidase A": "MLPKFLFLLAVSTINSHLALGDVPVQARQPGAALGMPPQMTTVPPFQGKDDELPSFK",
}

data = []
for name, seq in transmembrane_proteins.items():
    data.append({"name": name, "sequence": seq, "label": 1})

for name, seq in soluble_proteins.items():
    data.append({"name": name, "sequence": seq, "label": 0})

df_raw = pd.DataFrame(data)

print(f"✓ データセット作成完了")
print(f"  膜貫通型: {(df_raw['label']==1).sum()}")
print(f"  可溶性: {(df_raw['label']==0).sum()}")
print(f"\n最初の3行:")
print(df_raw.head(3))

## 5. 埋め込みベクトルの計算

In [ ]:
print("タンパク質埋め込みを計算中...")
embeddings = []

for idx, row in df_raw.iterrows():
    seq = row["sequence"]
    try:
        embedding = get_protein_embedding(seq, tokenizer, model, device)
        embeddings.append(embedding)
        if (idx + 1) % 3 == 0:
            print(f"  {idx+1}/{len(df_raw)} 完了", end="\r")
    except Exception as e:
        print(f"エラー (行 {idx}): {e}")
        embeddings.append(np.zeros(1280))

df_raw["embedding"] = embeddings
print(f"\n✓ 埋め込み計算完了: {len(embeddings)} タンパク質")

## 6. 埋め込みベクトルの可視化(t-SNE)

In [ ]:
print("t-SNE で次元削減中 (1分程度かかります)...")
X_embeddings = np.array(df_raw["embedding"].tolist())

tsne = TSNE(n_components=2, random_state=42, perplexity=5, n_iter=1000)
X_tsne = tsne.fit_transform(X_embeddings)

df_raw["tsne_x"] = X_tsne[:, 0]
df_raw["tsne_y"] = X_tsne[:, 1]

print("✓ 次元削減完了")

fig, ax = plt.subplots(figsize=(10, 7))
colors = {0: "#2E86AB", 1: "#A23B72"}
labels = {0: "可溶性", 1: "膜貫通型"}

for label in [0, 1]:
    mask = df_raw["label"] == label
    ax.scatter(
        df_raw.loc[mask, "tsne_x"],
        df_raw.loc[mask, "tsne_y"],
        c=colors[label],
        label=labels[label],
        s=100,
        alpha=0.7,
        edgecolors="black",
        linewidth=1
    )

ax.set_xlabel("t-SNE 成分 1")
ax.set_ylabel("t-SNE 成分 2")
ax.set_title("ProtBERT 埋め込みの2次元可視化")
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

## 7. 訓練・テストデータの分割

In [ ]:
X = np.array(df_raw["embedding"].tolist())
y = df_raw["label"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"✓ データ分割完了")
print(f"  訓練データ: {len(X_train)} サンプル")
print(f"    - 膜貫通型: {(y_train == 1).sum()}")
print(f"    - 可溶性: {(y_train == 0).sum()}")
print(f"  テストデータ: {len(X_test)} サンプル")
print(f"    - 膜貫通型: {(y_test == 1).sum()}")
print(f"    - 可溶性: {(y_test == 0).sum()}")
print(f"\n  埋め込みベクトル次元: {X.shape[1]}")

## 8. ロジスティック回帰による分類

In [ ]:
print("ロジスティック回帰を訓練中...")
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)
y_pred_proba_lr = lr.predict_proba(X_test)[:, 1]

print(f"\n✓ ロジスティック回帰の結果")
print(f"  訓練精度: {lr.score(X_train, y_train):.3f}")
print(f"  テスト精度: {lr.score(X_test, y_test):.3f}")
print(f"  ROC-AUC: {roc_auc_score(y_test, y_pred_proba_lr):.3f}")

print(f"\n分類レポート:")
print(classification_report(y_test, y_pred_lr,
                          target_names=["可溶性", "膜貫通型"]))

## 9. ランダムフォレストによる分類

In [ ]:
print("ランダムフォレストを訓練中...")
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_pred_proba_rf = rf.predict_proba(X_test)[:, 1]

print(f"\n✓ ランダムフォレストの結果")
print(f"  訓練精度: {rf.score(X_train, y_train):.3f}")
print(f"  テスト精度: {rf.score(X_test, y_test):.3f}")
print(f"  ROC-AUC: {roc_auc_score(y_test, y_pred_proba_rf):.3f}")

print(f"\n分類レポート:")
print(classification_report(y_test, y_pred_rf,
                          target_names=["可溶性", "膜貫通型"]))

## 10. モデル比較

In [ ]:
comparison_df = pd.DataFrame({
    "モデル": ["ロジスティック回帰", "ランダムフォレスト"],
    "訓練精度": [
        lr.score(X_train, y_train),
        rf.score(X_train, y_train)
    ],
    "テスト精度": [
        lr.score(X_test, y_test),
        rf.score(X_test, y_test)
    ],
    "ROC-AUC": [
        roc_auc_score(y_test, y_pred_proba_lr),
        roc_auc_score(y_test, y_pred_proba_rf)
    ]
})

print("\n📊 モデル比較")
print(comparison_df.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

x = np.arange(len(comparison_df))
width = 0.25

axes[0].bar(x - width, comparison_df["訓練精度"], width, label="訓練精度", color="skyblue")
axes[0].bar(x, comparison_df["テスト精度"], width, label="テスト精度", color="coral")
axes[0].bar(x + width, comparison_df["ROC-AUC"], width, label="ROC-AUC", color="lightgreen")
axes[0].set_ylabel("スコア")
axes[0].set_title("モデル性能比較")
axes[0].set_xticks(x)
axes[0].set_xticklabels(comparison_df["モデル"])
axes[0].legend()
axes[0].set_ylim([0, 1.05])
axes[0].grid(axis="y", alpha=0.3)

fpr_lr, tpr_lr, _ = roc_curve(y_test, y_pred_proba_lr)
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_pred_proba_rf)

axes[1].plot(fpr_lr, tpr_lr, label=f"LR (AUC={roc_auc_score(y_test, y_pred_proba_lr):.3f})", linewidth=2)
axes[1].plot(fpr_rf, tpr_rf, label=f"RF (AUC={roc_auc_score(y_test, y_pred_proba_rf):.3f})", linewidth=2)
axes[1].plot([0, 1], [0, 1], "k--", alpha=0.3)
axes[1].set_xlabel("偽陽性率")
axes[1].set_ylabel("真陽性率")
axes[1].set_title("ROC 曲線")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 11. 混同行列の可視化

In [ ]:
cm_lr = confusion_matrix(y_test, y_pred_lr)
cm_rf = confusion_matrix(y_test, y_pred_rf)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.heatmap(cm_lr, annot=True, fmt="d", cmap="Blues", ax=axes[0],
            xticklabels=["可溶性", "膜貫通型"],
            yticklabels=["可溶性", "膜貫通型"])
axes[0].set_title("ロジスティック回帰 - 混同行列")
axes[0].set_ylabel("実際のラベル")
axes[0].set_xlabel("予測ラベル")

sns.heatmap(cm_rf, annot=True, fmt="d", cmap="Greens", ax=axes[1],
            xticklabels=["可溶性", "膜貫通型"],
            yticklabels=["可溶性", "膜貫通型"])
axes[1].set_title("ランダムフォレスト - 混同行列")
axes[1].set_ylabel("実際のラベル")
axes[1].set_xlabel("予測ラベル")

plt.tight_layout()
plt.show()

## 12. ランダムフォレストの特徴量重要度

In [ ]:
feature_importance = pd.Series(
    rf.feature_importances_,
    index=[f"次元 {i}" for i in range(len(rf.feature_importances_))]
).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
feature_importance.head(20).plot(kind="barh", ax=ax, color="steelblue")
ax.set_xlabel("特徴量重要度")
ax.set_title("ProtBERT 埋め込みの特徴量重要度(上位20)")
plt.tight_layout()
plt.show()

print(f"\n💡 解釈:")
print(f"  - 上位の次元が分類に寄与している")
print(f"  - これらは『膜貫通型に特有なアミノ酸パターン』を表現")
print(f"  - 具体的には、疎水性アミノ酸(Leu, Val, Phe, Ile など)の集中")

## 13. 新しいタンパク質での予測

In [ ]:
new_proteins = {
    "Aquaporin-1 (膜蛋白)": "MSKGEQVSTAQESVQEAKS",
    "GFP (可溶性)": "MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTGKLPVWPT",
}

print("\n🔍 新しいタンパク質での予測\n")

for protein_name, sequence in new_proteins.items():
    embedding = get_protein_embedding(sequence, tokenizer, model, device)
    
    pred_lr = lr.predict([embedding])[0]
    prob_lr = lr.predict_proba([embedding])[0]
    
    pred_rf = rf.predict([embedding])[0]
    prob_rf = rf.predict_proba([embedding])[0]
    
    pred_label_lr = "膜貫通型" if pred_lr == 1 else "可溶性"
    pred_label_rf = "膜貫通型" if pred_rf == 1 else "可溶性"
    
    print(f"タンパク質: {protein_name}")
    print(f"  配列長: {len(sequence)} アミノ酸")
    print(f"  ロジスティック回帰: {pred_label_lr} (確信度: {max(prob_lr):.1%})")
    print(f"  ランダムフォレスト: {pred_label_rf} (確信度: {max(prob_rf):.1%})")
    print()

## 14. 実習のまとめ

In [ ]:
print("\n" + "="*70)
print("\t実習のまとめと発見")
print("="*70)

print(f"""
1️⃣ タンパク質言語モデル(ProtBERT)について:
   - アミノ酸配列を1280次元のベクトルに圧縮できた
   - このベクトルは『タンパク質の構造・機能上の特徴』を自動学習
   - GPT が人間の言語を学ぶように、
     ProtBERT は『タンパク質の言語』を習得している

2️⃣ 埋め込み空間の構造:
   - t-SNE の可視化で、膜貫通型と可溶性がおおよそ分離している
   - ProtBERT が『膜貫通領域の特徴(疎水性)』を
     埋め込み内に自動的に符号化していることを示唆

3️⃣ シンプルな分類器で十分:
   - 埋め込みが良質なので、複雑なニューラルネットワークは不要
   - ロジスティック回帰やランダムフォレストで高精度を達成

4️⃣ 特徴量重要度の解釈:
   - 上位の埋め込み次元が分類に寄与している
   - これらは『膜貫通型特有の物理化学的特性』を表現していると考えられる

5️⃣ 実務応用:
   - このアプローチは『ドメイン知識なしに』機能予測ができる
   - 既知のタンパク質が少ない新規遺伝子でも、
     配列だけから機能を推定可能
""")

print("="*70)

## 15. 発展課題

### 課題 1：より大規模なデータセット
UniProt の API を使用して数千のタンパク質を取得できます。

### 課題 2：他の分類タスク
- 酵素 vs 非酵素
- 信号ペプチド有無
- リン酸化サイト有無

### 課題 3：埋め込みの次元性削減
PCA で何次元あれば十分な精度が得られるか検証。

### 課題 4：他の言語モデルとの比較
- ESM-2 (Meta)
- ProtT5 (Google)

### 課題 5：特徴量の解釈
重要な埋め込み次元と、実際のアミノ酸特性の関連性を調査。